In [1]:
import pandas as pd
import numpy as np
from scipy.sparse import coo_matrix
from scipy.sparse.linalg import eigs

In [2]:
# Extracting data from Stanford University web.
url = "https://snap.stanford.edu/data/soc-sign-bitcoinotc.csv.gz"
columns = ['start', 'end','weight','time']
df = pd.read_csv(url,names = columns)

In [ ]:
# Creating a bijection between IDs and natural numbers. 
unique_nodes = pd.unique(df[['start','end']].values.ravel('K'))
node_map = {node_id: i for i, node_id in enumerate(unique_nodes)}
inverse_node_map = {i: node_id for i, node_id in enumerate(unique_nodes)}

df['i_start'] = df['start'].map(node_map)
df['i_end'] = df['end'].map(node_map)

N = len(unique_nodes)

In [4]:
# Computing weighted matrix using the weights of the directed edges.
I = df['i_start'].values
J = df['i_end'].values
w = df['weight'].values

A = coo_matrix((w, (I,J)), shape=(N,N))
print(f"Dimension (nodes): {N} x {N}")
print(f"Number of directed edges: {A.nnz}")

Dimension (nodes): 5881 x 5881
Number of directed edges: 35592


In [5]:
# Calculating eigenvalues and left and right eigenvectors from which we take only real parts.  
lambda_R, x = eigs(A,k=1,which='LM')
lambda_L, y = eigs(A.transpose(),k=1,which='LM')

u = np.real(x[:,0])
v = np.real(y[:,0])

# Assocciating each coordinate with its vertex ID and creating the A's Spectrum DataFrame. 
results =[]
for k in range(N):
    results.append((inverse_node_map[k],u[k],v[k]))

df_spectrum = pd.DataFrame(results, columns =['ID','Issuing Authority','Receiving Authority'])

df_negsink = df_spectrum.sort_values(by='Receiving Authority',ascending=True)
df_possink = df_spectrum.sort_values(by='Receiving Authority',ascending=False)

print("--- Lower End of the Receiving Spectrum ---")
print(df_negsink.head(10))
print("\n--- Upper End of the Receiving Spectrum ---")
print(df_possink.head(10))

--- Lower End of the Receiving Spectrum ---
        ID  Issuing Authority  Receiving Authority
1618  1810          -0.192886            -0.320444
3347  3897          -0.067031            -0.266026
3937  4635          -0.013100            -0.186356
870    905          -0.146129            -0.153198
3566  4172          -0.080538            -0.149453
1803  2045          -0.142609            -0.135179
1450  1565          -0.157387            -0.124218
1        1          -0.113055            -0.124201
3652  4291          -0.102818            -0.122969
2303  2642          -0.134586            -0.109396

--- Upper End of the Receiving Spectrum ---
        ID  Issuing Authority  Receiving Authority
3218  3744           0.152716             0.170288
1780  2017          -0.004190             0.139880
4009  4733           0.032458             0.114390
3967  4666           0.059373             0.109464
3966  4661           0.063749             0.109400
3850  4531           0.071285             0.

After analyzing the network topology, we can see that ID 3744 is a potential active scammer, as it has the lowest overall receiving score (-0.170) and also a strongly negative sending score (-0.152).
We will now investigate this user's interactions with the rest of the network to verify our hypothesis.

In [6]:
target_id = 3744

# Extracting 3744 related vertices.   
E_in = df[df['end'] == target_id]
wsum_in = E_in['weight'].sum()
numars_in = len(E_in)

E_out = df[df['start'] == target_id]
wsum_out = E_out['weight'].sum()
numars_out = len(E_out)

print(f"--- Local Topological Analysis of the {target_id} Vertex")
print(f"Cardinality of N^-(v): {numars_in} in-edges")
print(f"Weighted In-degree (deg_in): {wsum_in}")

print(f"Cardinality of N^+(v): {numars_out} out-edges")
print(f"Weighted Out-degree (deg_out): {wsum_out}")

print(f"\n --- E_in Set Extract for {target_id} vertex ---")
print(E_in[['start','weight']].head(10))

--- Local Topological Analysis of the 3744 Vertex
Cardinality of N^-(v): 81 in-edges
Weighted In-degree (deg_in): -675
Cardinality of N^+(v): 32 out-edges
Weighted Out-degree (deg_out): 80

 --- E_in Set Extract for 3744 vertex ---
       start  weight
19932   2962      10
19945   1802     -10
19946   1363     -10
19947   2658     -10
19949   2296     -10
19951   2647     -10
19957   2045     -10
19959   2028     -10
19999   2252      -5
20013   1948     -10


Since the vertex with ID 2962 stands out from the rest for giving such positive confidence to 3744, we studied the interaction between these two users and other possible suspects.

In [7]:
# Compute the adjacency matrix for cases of maximum trust.
df_trust = df[df['weight'] == 10].copy()
M_files, M_columns = df_trust['i_start'], df_trust['i_end']
M_values = np.ones(len(df_trust))

M = coo_matrix((M_values,(M_files,M_columns)), shape=(N,N))
M = M.tocsr()

i_target = node_map[3744]
i_suspect = node_map[2962]

# Calculating 2-step paths. 
M2 = M.dot(M)

go_ar = M[i_target,i_suspect]
back_ar = M[i_suspect,i_target]

cycles_target = M2[i_target, i_target]
cycles_suspect = M2[i_suspect, i_suspect]

print("--- Evaluating adjacency matrix M ---")
print(f"Edge Existence 2962 -> 3744 (m_ij): {int(go_ar)}")
print(f"Edge Existence 3744 -> 2962 (m_ij): {int(back_ar)}")

print("--- Evaluating 2-Step Paths Matrix M^2 ---")
print(f"2-Step Cycles for 3744 node: {int(cycles_target)}")
print(f"2-Step Cycles for 2962 node: {int(cycles_suspect)}")

--- Evaluating adjacency matrix M ---
Edge Existence 2962 -> 3744 (m_ij): 1
Edge Existence 3744 -> 2962 (m_ij): 1
--- Evaluating 2-Step Paths Matrix M^2 ---
2-Step Cycles for 3744 node: 4
2-Step Cycles for 2962 node: 1


In [8]:
# Computing Hadamard product (M times M^t)
C = M.multiply(M.transpose())

file_target = C[i_target, :]

i_party = file_target.nonzero()[1]
suspects = []
for i in i_party: 
    x = inverse_node_map[i]
    suspects.append(x)

weights_intersection = file_target.data

print(f"--- Collusion Ring Analysis for the Scammer {target_id} ---")

if len(i_party) == 0:
    print("Empty subspace. There is no bidirectional collusion.")
else:
    for i,idx in enumerate(suspects):
        weight = weights_intersection[i]
        print(f"Bidirectional accomplice found: original ID {suspects[i]}")
        print(f"-> Singularity Magnitude: {int(weight)}")

--- Collusion Ring Analysis for the Scammer 3744 ---
Bidirectional accomplice found: original ID 2962
-> Singularity Magnitude: 1
Bidirectional accomplice found: original ID 3756
-> Singularity Magnitude: 1
Bidirectional accomplice found: original ID 3759
-> Singularity Magnitude: 1
Bidirectional accomplice found: original ID 3760
-> Singularity Magnitude: 1


In [9]:
print("---Strict Topological Audit---")

for x in suspects:
    i = node_map[x]

    m_go = M[i, i_target]
    m_back = M[i_target, i]

    c = m_go * m_back
    cm = C[i_target,i]

    print(f"ID {x}")
    print(f"  M[{x} -> 3744] (in) = {int(m_go)}")
    print(f"  M[3744 -> {x}] (out) = {int(m_back)}")
    print(f"  Theoretic Hadamard (in * out) = {int(c)}")
    print(f"  Actual value stored in C = {int(cm)}\n")

---Strict Topological Audit---
ID 2962
  M[2962 -> 3744] (in) = 1
  M[3744 -> 2962] (out) = 1
  Theoretic Hadamard (in * out) = 1
  Actual value stored in C = 1

ID 3756
  M[3756 -> 3744] (in) = 1
  M[3744 -> 3756] (out) = 1
  Theoretic Hadamard (in * out) = 1
  Actual value stored in C = 1

ID 3759
  M[3759 -> 3744] (in) = 1
  M[3744 -> 3759] (out) = 1
  Theoretic Hadamard (in * out) = 1
  Actual value stored in C = 1

ID 3760
  M[3760 -> 3744] (in) = 1
  M[3744 -> 3760] (out) = 1
  Theoretic Hadamard (in * out) = 1
  Actual value stored in C = 1

